# Stardox Email Scraper

Takes a list of GitHub usernames and scrapes their public email addresses using a headless browser.

**How it works:**
1. Spins up headless Chromium via Playwright (downloads its own browser — no system Chrome needed)
2. Visits each user's GitHub profile
3. Looks for email in the profile sidebar (JS-rendered)
4. If no email on profile, checks their commit history (.patch files)
5. Outputs username:email pairs as a downloadable CSV

In [ ]:
# Install Playwright + its own bundled Chromium (does NOT use system Chrome)
!pip install -q playwright nest_asyncio pandas tqdm
!playwright install chromium
!playwright install-deps chromium

In [ ]:
import re
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright

nest_asyncio.apply()

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
IGNORE_PATTERNS = ['noreply', 'users.noreply.github.com', 'github.com', 'githubusercontent']


def is_valid_email(email):
    if not email:
        return False
    email_lower = email.lower()
    for pattern in IGNORE_PATTERNS:
        if pattern in email_lower:
            return False
    return True


async def start_browser(github_cookie=None):
    """Launch headless Chromium via Playwright async API."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)

    if github_cookie:
        context = await browser.new_context()
        await context.add_cookies([{
            'name': 'user_session',
            'value': github_cookie,
            'domain': '.github.com',
            'path': '/',
            'secure': True,
        }])
        page = await context.new_page()
        print('Browser started with GitHub session!')
    else:
        page = await browser.new_page()
        print('Browser started (anonymous — profile emails will be hidden)')

    return pw, browser, page


async def stop_browser(pw, browser):
    """Clean up browser and playwright."""
    try:
        await browser.close()
    except Exception:
        pass
    try:
        await pw.stop()
    except Exception:
        pass


async def scrape_email_from_profile(page, username):
    """Visit GitHub profile and extract email from page text."""
    try:
        await page.goto(f'https://github.com/{username}', wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(2000)

        body_text = await page.inner_text('body')

        emails = EMAIL_RE.findall(body_text)
        for email in emails:
            if is_valid_email(email):
                return email

        source = await page.content()
        mailto_matches = re.findall(r'mailto:([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', source)
        for email in mailto_matches:
            if is_valid_email(email):
                return email

    except Exception:
        pass

    return None


async def scrape_email_from_commits(page, username):
    """Get email from user's commit .patch files."""
    try:
        await page.goto(f'https://github.com/{username}?tab=repositories&type=source',
                         wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(1000)

        repo_elements = await page.query_selector_all('a[itemprop="name codeRepository"]')
        repo_names = []
        for el in repo_elements[:3]:
            name = await el.inner_text()
            repo_names.append(name.strip())

        if not repo_names:
            return None

        for repo_name in repo_names:
            try:
                await page.goto(
                    f'https://github.com/{username}/{repo_name}/commits?author={username}',
                    wait_until='networkidle', timeout=20000)
                await page.wait_for_timeout(1000)

                commit_links = await page.query_selector_all(
                    f'a[href*="/{username}/{repo_name}/commit/"]')

                for commit_link in commit_links[:5]:
                    href = await commit_link.get_attribute('href')
                    if not href or '/commit/' not in href:
                        continue

                    label = (await commit_link.get_attribute('aria-label')) or ''
                    text = (await commit_link.inner_text()) or ''
                    if 'merge' in label.lower() or 'merge' in text.lower():
                        continue

                    if href.startswith('/'):
                        href = 'https://github.com' + href

                    await page.goto(href + '.patch', timeout=15000)
                    await page.wait_for_timeout(1000)

                    page_text = await page.content()

                    from_match = re.search(r'From:.*?<([^>]+@[^>]+)>', page_text)
                    if from_match:
                        email = from_match.group(1)
                        if is_valid_email(email):
                            return email

                    emails = EMAIL_RE.findall(page_text[:3000])
                    for email in emails:
                        if is_valid_email(email):
                            return email

            except Exception:
                continue

    except Exception:
        pass

    return None


async def scrape_email(page, username):
    """Try profile first, then commits."""
    email = await scrape_email_from_profile(page, username)
    if email:
        return email
    return await scrape_email_from_commits(page, username)


print('Functions loaded. Ready to scrape.')

In [ ]:
# ===========================================
# GITHUB SESSION COOKIE (required to see profile emails)
#
# How to get it:
# 1. Log into github.com in your browser
# 2. Open DevTools (F12) -> Application -> Cookies -> github.com
# 3. Find the "user_session" cookie and copy its value
# ===========================================

GITHUB_COOKIE = ""  # paste your user_session cookie value here

# ===========================================
# PASTE YOUR USERNAMES BELOW (one per line)
# ===========================================

usernames_input = """
bnbarak
torvalds
sindresorhus
"""

# ===========================================

usernames = [u.strip() for u in usernames_input.strip().split('\n') if u.strip()]
print(f'Loaded {len(usernames)} usernames')
if GITHUB_COOKIE:
    print('GitHub cookie provided — will see profile emails')
else:
    print('No GitHub cookie — will only get emails from commit history')

In [ ]:
# Run the scraper
pw, browser, page = await start_browser(github_cookie=GITHUB_COOKIE if GITHUB_COOKIE else None)
results = []
found_count = 0

try:
    for username in tqdm(usernames, desc='Scraping emails'):
        email = await scrape_email(page, username)
        results.append({'username': username, 'email': email})

        if email:
            found_count += 1
            print(f'  ✓ {username} -> {email}')
        else:
            print(f'  ✗ {username} -> not found')

        await page.wait_for_timeout(1000)  # be polite
finally:
    await stop_browser(pw, browser)

print(f'\nDone! Found {found_count}/{len(usernames)} emails')

In [ ]:
# Results
df = pd.DataFrame(results)
print(f'Total: {len(df)}')
print(f'With email: {df["email"].notna().sum()}')
print(f'Without email: {df["email"].isna().sum()}')
print()

# Show all results
display(df)

# Save and download CSV
csv_filename = 'stargazer_emails.csv'
df.to_csv(csv_filename, index=False)

from google.colab import files
files.download(csv_filename)
print(f'\nDownloading {csv_filename}...')